# Projet Final - Forage de données (8INF436)
**Sujet :** Prédire le statut de menace d'une espèce animale  
**Groupe :** Frantxa Cabrejos, Loup-Djabril Le Bivic, Nathan Razafindratsima  
**Dataset :** World Wildlife Species Data - Kaggle (10 000+ observations)  
**Problème :** Classification multi-classes : Non menacé / Menacé / Disparu

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

## 1. Chargement du dataset

Source : https://www.kaggle.com/datasets/chirayurijal/worldwildlifespeciesdata  
Le fichier CSV doit etre dans le meme dossier que ce notebook.

In [ ]:
import glob

csv_files = glob.glob('*.csv')
print('Fichiers CSV trouves :', csv_files)

CSV_PATH = csv_files[0] if csv_files else 'species_data.csv'
df = pd.read_csv(CSV_PATH, low_memory=False)

print(f'\nDataset charge avec succes.')
print(f'Nombre d observations : {df.shape[0]}')
print(f'Nombre de colonnes    : {df.shape[1]}')

## 2. Apercu des donnees

In [ ]:
# premieres lignes
df.head(10)

In [ ]:
# liste des colonnes
print(f'Colonnes disponibles ({len(df.columns)}) :')
for col in df.columns:
    print(f'  {col}')

In [ ]:
# types et valeurs non-nulles par colonne
df.info()

In [ ]:
# statistiques descriptives
df.describe(include='all')

## 3. Analyse des valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'nb_manquants': missing,
    'pourcentage (%)': missing_pct
}).sort_values('pourcentage (%)', ascending=False)

avec_manquants = missing_df[missing_df['nb_manquants'] > 0]

if len(avec_manquants) > 0:
    print(f'{len(avec_manquants)} colonne(s) avec valeurs manquantes :\n')
    print(avec_manquants.to_string())
else:
    print('Aucune valeur manquante dans le dataset.')

In [ ]:
# visualisation des valeurs manquantes
if len(avec_manquants) > 0:
    plt.figure(figsize=(11, 5))
    avec_manquants['pourcentage (%)'].plot(
        kind='bar',
        color='steelblue',
        edgecolor='black'
    )
    plt.axhline(60, color='red', linestyle='--', label='Seuil 60% (suppression prevue)')
    plt.title('Pourcentage de valeurs manquantes par colonne', fontsize=13)
    plt.ylabel('% de valeurs manquantes')
    plt.xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Rien a afficher : pas de valeurs manquantes.')

## 4. Identification et distribution de la variable cible

In [ ]:
# detection automatique de la colonne IUCN
mots_cles = ['category', 'status', 'iucn', 'conservation', 'threat', 'red']
candidates = [c for c in df.columns if any(kw in c.lower() for kw in mots_cles)]

print('Colonnes candidates pour la variable cible :')
for col in candidates:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

In [ ]:
# adapter si besoin selon ce qui s'affiche ci-dessus
TARGET_COL = candidates[0] if candidates else 'category'
print(f'Variable cible retenue : {TARGET_COL}')

In [ ]:
# regroupement des categories IUCN en 3 classes
#   LC, NT -> Non menace
#   VU, EN, CR -> Menace
#   EW, EX -> Disparu
#   DD, NE -> supprimes (pas classifiables)

IUCN_MAPPING = {
    'LC'   : 'Non menace',
    'NT'   : 'Non menace',
    'LR/lc': 'Non menace',
    'LR/cd': 'Non menace',
    'LR/nt': 'Non menace',
    'VU'   : 'Menace',
    'EN'   : 'Menace',
    'CR'   : 'Menace',
    'EW'   : 'Disparu',
    'EX'   : 'Disparu',
}

df['label'] = df[TARGET_COL].map(IUCN_MAPPING)

nb_avant = len(df)
df_filtre = df[df['label'].notna()].copy()
nb_apres = len(df_filtre)

print(f'Lignes retirees (DD, NE, etc.) : {nb_avant - nb_apres}')
print(f'Observations conservees        : {nb_apres}')
print(f'\nDistribution des 3 classes :')
print(df_filtre['label'].value_counts())

In [ ]:
# visualisation de la distribution des classes
counts = df_filtre['label'].value_counts()
colors = ['#4caf50', '#f44336', '#9e9e9e']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# barplot
bars = axes[0].bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='black')
axes[0].set_title('Nombre d especes par classe')
axes[0].set_ylabel('Nombre d especes')
axes[0].set_xlabel('Statut de menace')
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        val + 30,
        str(val),
        ha='center',
        fontweight='bold'
    )

# camembert
axes[1].pie(
    counts.values,
    labels=counts.index,
    colors=colors[:len(counts)],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Repartition en pourcentage')

plt.suptitle('Distribution de la variable cible (statut de menace IUCN)', fontsize=13)
plt.tight_layout()
plt.show()

# signaler le desequilibre
ratio = counts.max() / counts.min()
print(f'Ratio de desequilibre (max/min) : {ratio:.1f}x')
if ratio > 3:
    print('Classes desequilibrees detectees.')
    print('Traitement prevu dans le preprocessing : SMOTE (Synthetic Minority Oversampling).')

## 5. Analyse des types de variables

In [ ]:
features = df_filtre.drop(columns=[TARGET_COL, 'label'], errors='ignore')

cat_cols = features.select_dtypes(include='object').columns.tolist()
num_cols = features.select_dtypes(include=np.number).columns.tolist()

print(f'Variables numeriques ({len(num_cols)}) :')
for c in num_cols:
    print(f'  {c}')

print(f'\nVariables categorielles ({len(cat_cols)}) :')
for c in cat_cols:
    nb_uniques = features[c].nunique()
    print(f'  {c:40s} ({nb_uniques} valeurs uniques)')

In [ ]:
# repartition globale numerique vs categoriel
plt.figure(figsize=(5, 4))
plt.bar(
    ['Numeriques', 'Categorielles'],
    [len(num_cols), len(cat_cols)],
    color=['#2196f3', '#ff9800'],
    edgecolor='black'
)
plt.title('Repartition des types de variables')
plt.ylabel('Nombre de colonnes')
for i, v in enumerate([len(num_cols), len(cat_cols)]):
    plt.text(i, v + 0.2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Bilan de l'exploration et prochaines etapes

### Ce qu'on a fait
- Dataset charge : 10 000+ observations, colonnes variees (taxonomie, habitat, geographie)
- Variable cible identifiee et recodee en 3 classes (Non menace / Menace / Disparu)
- Valeurs manquantes analysees et visualisees
- Types de variables identifies (numeriques et categorielles)
- Desequilibre de classes quantifie

### Prochaines etapes (preprocessing)
1. Suppression des colonnes avec trop de valeurs manquantes (seuil 60%)
2. Suppression des colonnes texte libre a haute cardinalite (identifiants)
3. Imputation des valeurs manquantes (mediane pour les numeriques, mode pour les categorielles)
4. Encodage des variables categorielles (Label Encoding)
5. Normalisation MinMax
6. Traitement du desequilibre des classes avec SMOTE

### Modeles prevus
- Random Forest
- SVM (Support Vector Machine)
- Gradient Boosting

### Strategie de validation
- Split train/test 80/20 (stratifie)
- Stratified K-Fold k=5 pour la validation croisee